In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

df = pd.read_csv("UrbanCabFare.csv")

print(f"Dataset shape: {df.shape}")
display(df.head())

Dataset shape: (1000, 16)


,User_ID,Driver_Name,City,Ride_Date,Hour_of_Day,Day_of_Week,Distance_km,Traffic_Level,Type_of_vehicle,No_of_active_drivers,Ride_Requests,Demand_Supply_Ratio,Trip_Duration,Base_Fare,Surge_Multiplier,Final_Fare
0,USER-21819-Bcbf,Patrick Sanchez,Pune,05-11-2025,0,Friday,26.210,Low,Hatchback,36,67,1.860,60,354.520,1.300,460.880
1,USER-79402-SBvr,Meredith Barnes,Mumbai,13-10-2025,2,Friday,21.080,High,Auto,9,17,1.890,50,282.960,1.300,367.850
2,USER-40781-yfJs,Ian Cooper,Bengaluru,16-06-2026,6,Sunday,25.340,High,Hatchback,94,149,1.590,75,344.080,1.300,447.300
3,USER-41316-rDOx,Amber Perez,Mumbai,15-10-2025,0,Monday,26.800,High,Auto,25,188,7.520,76,351.600,1.600,562.560
4,USER-76483-RuXX,Kimberly Sanchez,Chennai,06-09-2025,10,Friday,4.480,High,SUV,53,34,0.640,18,133.760,1.000,133.760


In [3]:
schema = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique": df.nunique()
})

display(schema)

print(f"\nExact duplicate rows: {df.duplicated().sum():,}")

,dtype,missing,missing_pct,unique
User_ID,object,0,0.000,1000
Driver_Name,object,15,1.500,976
City,object,0,0.000,6
Ride_Date,object,0,0.000,340
Hour_of_Day,int64,0,0.000,24
Day_of_Week,object,0,0.000,7
Distance_km,float64,0,0.000,863
Traffic_Level,object,0,0.000,4
Type_of_vehicle,object,0,0.000,5
No_of_active_drivers,int64,0,0.000,96



Exact duplicate rows: 0


In [5]:
for col in df.columns:
    print(f"{col:25s} | unique = {df[col].nunique():,}")

User_ID                   | unique = 1,000
Driver_Name               | unique = 976
City                      | unique = 6
Ride_Date                 | unique = 340
Hour_of_Day               | unique = 24
Day_of_Week               | unique = 7
Distance_km               | unique = 863
Traffic_Level             | unique = 4
Type_of_vehicle           | unique = 5
No_of_active_drivers      | unique = 96
Ride_Requests             | unique = 190
Demand_Supply_Ratio       | unique = 517
Trip_Duration             | unique = 114
Base_Fare                 | unique = 949
Surge_Multiplier          | unique = 11
Final_Fare                | unique = 987


In [6]:
fare_check = (
    df["Final_Fare"]
    - df["Base_Fare"] * df["Surge_Multiplier"]
)

print(f"Maximum absolute difference: {fare_check.abs().max():.6f}")
print(f"Mean absolute difference:    {fare_check.abs().mean():.6f}")
print(f"Correlation:                 {df['Base_Fare'].corr(df['Final_Fare']):.6f}")

Maximum absolute difference: 0.004000
Mean absolute difference:    0.001546
Correlation:                 0.870428


In [7]:
print(f"Exact duplicate rows: {df.duplicated().sum():,}")

numeric_cols = [
    "Distance_km",
    "No_of_active_drivers",
    "Ride_Requests",
    "Demand_Supply_Ratio",
    "Trip_Duration",
    "Surge_Multiplier",
    "Final_Fare"
]

df[numeric_cols].describe().T

Exact duplicate rows: 0


,count,mean,std,min,25%,50%,75%,max
Distance_km,"1,000.000",17.911,9.634,1.010,9.568,18.180,25.898,34.990
No_of_active_drivers,"1,000.000",52.520,27.473,5.000,28.000,54.000,76.000,100.000
Ride_Requests,"1,000.000",108.047,55.394,10.000,60.000,110.500,156.000,200.000
Demand_Supply_Ratio,"1,000.000",3.569,4.734,0.100,1.157,2.040,4.022,39.400
Trip_Duration,"1,000.000",55.165,27.947,6.000,32.000,54.000,76.250,122.000
Surge_Multiplier,"1,000.000",1.436,0.343,1.000,1.000,1.500,1.700,2.000
Final_Fare,"1,000.000",370.238,199.233,32.540,208.205,338.580,493.768,939.760


In [8]:
df["Demand_Supply_Ratio"].quantile(
    [0, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
)

0.000    0.100
0.500    2.040
0.750    4.022
0.900    7.645
0.950   11.943
0.990   26.000
1.000   39.400
Name: Demand_Supply_Ratio, dtype: float64

In [9]:
ratio_check = (
    df["Ride_Requests"] / df["No_of_active_drivers"]
)

comparison = pd.DataFrame({
    "recorded_ratio": df["Demand_Supply_Ratio"],
    "calculated_ratio": ratio_check,
    "difference": df["Demand_Supply_Ratio"] - ratio_check
})

comparison["difference"].abs().describe()

count   1,000.000
mean        0.002
std         0.002
min         0.000
25%         0.001
50%         0.002
75%         0.004
max         0.005
Name: difference, dtype: float64

In [10]:
model_numeric = [
    "Hour_of_Day",
    "Distance_km",
    "No_of_active_drivers",
    "Ride_Requests",
    "Demand_Supply_Ratio",
    "Trip_Duration",
    "Surge_Multiplier",
    "Final_Fare"
]

df[model_numeric].corr().round(3)

,Hour_of_Day,Distance_km,No_of_active_drivers,Ride_Requests,Demand_Supply_Ratio,Trip_Duration,Surge_Multiplier,Final_Fare
Hour_of_Day,1.000,0.067,-0.021,0.003,0.040,0.077,0.010,0.069
Distance_km,0.067,1.000,0.001,-0.011,-0.009,0.947,0.011,0.854
No_of_active_drivers,-0.021,0.001,1.000,-0.053,-0.611,0.007,-0.543,-0.237
Ride_Requests,0.003,-0.011,-0.053,1.000,0.413,0.013,0.636,0.283
Demand_Supply_Ratio,0.040,-0.009,-0.611,0.413,1.000,-0.013,0.487,0.202
Trip_Duration,0.077,0.947,0.007,0.013,-0.013,1.000,0.021,0.819
Surge_Multiplier,0.010,0.011,-0.543,0.636,0.487,0.021,1.000,0.456
Final_Fare,0.069,0.854,-0.237,0.283,0.202,0.819,0.456,1.000


## Data Audit Summary

- The dataset contains 1,000 ride records and 16 variables.
- `Final_Fare` was selected as the modelling target.
- `User_ID` and `Driver_Name` were excluded as non-generalizable identifier fields.
- `Base_Fare` was excluded because `Final_Fare` is approximately equal to
  `Base_Fare × Surge_Multiplier`, creating direct target leakage.
- Missing values occur only in the excluded `Driver_Name` field and therefore
  require no imputation.
- Numerical variables showed plausible operating ranges; no generic outlier
  removal was considered necessary.
- `Demand_Supply_Ratio` was verified against ride requests and active drivers
  and retained as a valid operational feature.
- Strong correlation between `Distance_km` and `Trip_Duration` indicates
  potential multicollinearity, which will be formally assessed during OLS
  diagnostics and provides motivation for regularized regression.